In [1]:
import os
from pathlib import Path
# Change cwd to the project root (parent of 'notebooks/')
os.chdir(Path.cwd().parent)
Path.cwd()

PosixPath('/Users/jbrandt/code/birddog')

In [2]:
# uncomment to use hosted db
#del os.environ["BIRDDOG_USE_LOCAL_NOCODB"]
os.environ.get("BIRDDOG_USE_LOCAL_NOCODB")

In [3]:
import json
from urllib.parse import urlparse, unquote
import mwparserfromhell

from birddog.runtime import Runtime
from birddog.database import Database
from birddog.database_updater import (
    DatabaseUpdater,
    DatabaseUpdateManager,
    )
from birddog.tracker import PageChangeLog
from birddog.utility import fetch_url, transliterate
from birddog.wiki import (
    WIKI_NAMESPACE,
    API_URL,
    expand_link_target,
    canonicalize_title,
    classify_page,
    page_name,
    )

2026-01-21 15:39:19,890 [INFO] Using local folder /Users/jbrandt/code/birddog/.cache for storage.
2026-01-21 15:39:19,987 [INFO] Translation is enabled. Using GCP translator
2026-01-21 15:39:19,988 [INFO] Using Google Cloud translation API
2026-01-21 15:39:19,988 [INFO] GoogleCloudTranslator using REST API
2026-01-21 15:39:20,267 [INFO] Using aws nocodb api: http://nocodb-env.eba-xhmfyydr.us-east-2.elasticbeanstalk.com


In [4]:
def clear_db():
    updater = runtime._database_update_manager._updater
    ids=updater._db.get_all_ids("Documents")
    updater._db.delete("Documents", ids)
    ids=updater._db.get_all_ids("Pages")
    updater._db.delete("Pages", ids)
def clear_alerts():
    updater = runtime._database_update_manager._updater
    updater.clear_alerts()

In [5]:
import random
def deterministic_shuffle(items, seed=42):
    rng = random.Random(seed)   # independent RNG instance
    items = list(items)         # avoid mutating caller’s list
    rng.shuffle(items)
    return items

In [6]:
runtime = Runtime()
#mgr = DatabaseUpdateManager(Runtime())
#updater = DatabaseUpdater(Runtime())

2026-01-21 15:39:21,300 [INFO] PageUpdateManager.init(): detect_environment==local
2026-01-21 15:39:21,932 [INFO] fetch_url: 1 requests in last 60s → 0.02 req/s
2026-01-21 15:39:22,579 [INFO] KillSwitch: loading thresholds from resources/kill_thresholds.json
2026-01-21 15:39:22,582 [INFO] Runtime: truncating log history before 2025-11-22 22:39:22.582125+00:00


In [ ]:
with open("var/title_sample.json") as file:
    title_list = json.loads(file.read())
title_list = deterministic_shuffle(sorted(list(set(title_list))))

In [ ]:
#clear_db()

In [ ]:
#clear_alerts()

In [ ]:
runtime.database_update_enabled

In [17]:
#title = "ДАЧкО/8/2/330"
#title = "ДАЧкО/388/1/30"
#title = "ДАОО/359/1/228"
#title = "ДАЧгО/Р-8997/1/120"
title = "ДАВоО/35/9/318"

In [18]:
runtime.update_to_database(title)

2026-01-21 15:52:28,650 [INFO] TaskManager starting task 01KFHC4RK41KNRWRHW4BSKRGF4
2026-01-21 15:52:28,657 [INFO] Updater: accessing wiki page info for 1 pages
2026-01-21 15:52:28,984 [INFO] fetch_url: 1 requests in last 60s → 0.02 req/s
2026-01-21 15:52:30,399 [INFO] Updater: analyzing page links
2026-01-21 15:52:31,784 [INFO] Updater: linking child pages
2026-01-21 15:52:33,863 [INFO] Updater: accessing linked document metadata
2026-01-21 15:52:34,080 [INFO] missing title: File:ДАВоО_35-9-318._Метричні_книги_Св._Георгіївської_церкви_с._Седлище_(1899-1900)_та_Св._Преображенської_церкви_м._Вижва_(1897)_Ковельського_повіту.pdf, File:ДАВоО 35-9-318. Метричні книги Св. Георгіївської церкви с. Седлище (1899-1900) та Св. Преображенської церкви м. Вижва (1897) Ковельського повіту.pdf, {'ns': 6, 'title': 'File:ДАВоО 35-9-318. Метричні книги Св. Георгіївської церкви с. Седлище (1899-1900) та Св. Преображенської церкви м. Вижва (1897) Ковельського повіту.pdf', 'missing': '', 'imagerepository':

In [ ]:
updater = DatabaseUpdater(runtime=runtime)

In [ ]:
from birddog.store import get_key_value_store
store = get_key_value_store()
int(store.get("DB Loader", "cursor"))

In [ ]:
#store.remove_all("DB Loader")

In [ ]:
from time import sleep

import json
from birddog.runtime import Runtime
from birddog.database import Database
from birddog.database_updater import DatabaseUpdater
from birddog.tracker import PageChangeLog
from birddog.store import get_key_value_store

updater = DatabaseUpdater(runtime=Runtime())
store = get_key_value_store()
change_log = PageChangeLog()
changes = change_log.get()
update_titles = sorted([title.replace("Архів:", "") for title in changes.keys()])

#def update_batch(updater, titles):
#    updater.update_records(titles)
#    updater.start_translation()


In [ ]:
update_titles = deterministic_shuffle(update_titles)

In [ ]:
len(update_titles)

In [ ]:
#runtime.update_to_database(update_titles[:30])

In [ ]:
db = Database()

In [ ]:
docs = db.scan_all("Documents")

In [ ]:
len(docs)

In [ ]:
docs[0]

In [ ]:
[d for d in docs if not d.get("owning_page")]

In [ ]:
doc_ids = [doc["Id"] for doc in docs if not doc["availability"] and doc["source"]]

In [ ]:
pid_map = {}
for did in doc_ids:
    pid = db.get_links("Documents", "owning_page", did)
    pid_map[did] = pid

In [ ]:
pid_map

In [ ]:
pid_updates = []
for ids in pid_map.values():
    pid_updates.extend(ids)
pid_updates = sorted(list(set(pid_updates)))

In [ ]:
pid_updates

In [ ]:
page_records = db.read("Pages", pid_updates)

In [ ]:
len(page_records)

In [ ]:
page_titles = [rec["title"] for rec in page_records]

In [ ]:
runtime.update_to_database(page_titles)

In [ ]:
db._host

In [ ]:
db = Database()

In [ ]:
db._host

In [ ]:
runtime

In [ ]:
u = DatabaseUpdater(runtime=runtime)

In [ ]:
u._db._host

In [ ]:
t = u._collect_translations()

In [ ]:
len(t)

In [ ]:
t